> **🔒 SOLUTIONS COPY** — private, not on the public remote. Exercise 1 and the falcon config are filled in, and the guided baseline (which the student notebook poses as an open-ended problem) is worked through to a posterior. See `hackathon_solutions/README.md`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chreissel/sbi-tutorial-iaifi26/blob/main/03_hackathon_stellar_streams.ipynb)

# Notebook 3 — Milky-Way stellar streams (hackathon)

**IAIFI Summer School · simulation-based inference · full-day hackathon.**

Notebooks 1 and 2 built the machinery: a flow posterior, a learned data
embedding, and `falcon`'s "the model is a graph in YAML" workflow. This notebook
points all of it at a real astrophysics problem and then **hands you the keys** —
it is deliberately short. You get a working forward model and one worked warm-up;
the inference itself is yours to build.

A **stellar stream** is what is left when a star cluster or dwarf galaxy is
torn apart by the Milky Way's tides: its stars stretch into a thin ribbon
tracing the orbit. The shape, width, and kinematics of that ribbon encode the
progenitor (its mass and age) *and* the Galactic potential the stream fell
through — which is why streams are one of our best handles on the distribution
of **dark matter**. We use [`sstrax`](https://github.com/undark-lab/sstrax), a
`jax` simulator of the **GD1** stream, wrapped for `falcon` exactly the way
notebook 2 wrapped the gravitational-wave chirp.

> ### ▶️ Run this cell first
> **Runtime → Change runtime type → T4 GPU** helps the flow/CNN training, but
> the simulator itself is CPU-bound `jax`, so a CPU runtime works too. Like
> notebook 2 we **clone** the repo (falcon reads configs and writes samples to
> disk) and install the physics package `sstrax`.
>
> **If you forked or renamed this repository**, change `chreissel/sbi-tutorial-iaifi26`
> below and in the Colab badge above to your own `USER/REPO`.
>
> The install pulls `jax` + `diffrax`; `sstrax` is a git package (not on PyPI).
> The pinned `diffrax` silences a deprecation warning it triggers.
>
> **Expect red text.** `pip` will print `dependency conflicts` about
> `ibis-framework`, `numba`, and `inflect` — Colab preinstalls this notebook
> never imports. They are **safe to ignore**; the install succeeded. If a
> *later* cell ever complains that numpy's version changed, that is Colab
> holding on to its old numpy: **Runtime → Restart session**, then run the
> notebook again from the top.

In [ ]:
# SOLUTIONS COPY — run from the repo root with the streams env already set up
# (see hackathon_solutions/README.md). The clone/install below is what the
# student notebook runs on a fresh Colab:
#   !git clone -q https://github.com/chreissel/sbi-tutorial-iaifi26.git
#   %cd sbi-tutorial-iaifi26
#   !pip install -q -r requirements.txt
#   !pip install -q "diffrax==0.6.1" "git+https://github.com/undark-lab/sstrax.git"

In [ ]:
import warnings; warnings.filterwarnings("ignore")   # sstrax/diffrax are chatty
import time
import numpy as np
import matplotlib.pyplot as plt

import streams_model as sm        # the simulator, wrapped for falcon
import streams_plotting as sp     # matplotlib-only plotting helpers

SEED = 0
np.random.seed(SEED)

---
## 0 — A stellar stream, in one simulator call

`sstrax.simulate_stream` takes **16 parameters** (`sm.PRIOR_LIST`) — the
progenitor's present-day position and velocity, its disruption age and mass,
and eight tidal-stripping "micro" parameters — and integrates the disrupting
cluster forward, returning the phase-space coordinates of the stream stars in
the Milky-Way (`halo`) frame:

`stars.shape == (N_stars, 6)` = (x, y, z, vx, vy, vz).

`N_stars` is **not fixed** — older or heavier progenitors shed more stars (a few
hundred to ~1000 here). The **first** call spends ~13 s compiling the `jax`
simulator; every call after that is fast.

In [ ]:
# The GD1 fiducial (sm.TRUE_VALUES): the parameters the observation, the GD1
# binning window, and every task below are built around. (sstrax's own
# Parameters() defaults are a *different* point whose stream lands just outside
# the GD1 window, so we use the fiducial from the start.)
params = sm.params_from_vector([sm.TRUE_VALUES[k] for k in sm.TRUE_VALUES],
                               list(sm.TRUE_VALUES))

t0 = time.time()
stars = np.asarray(sm.sstrax.simulate_stream(key=sm.jax.random.PRNGKey(0), params=params))
print(f"first call : {time.time()-t0:5.1f} s  (includes ~13 s JIT compile)")
t0 = time.time()
_ = np.asarray(sm.sstrax.simulate_stream(key=sm.jax.random.PRNGKey(1), params=params))
print(f"second call: {time.time()-t0:5.2f} s")
print("stars.shape:", stars.shape, "-> (N_stars, 6)")

**Two ways to look at one stream.** On the left, the stars in the Galactic
(`halo`) frame — the physical ribbon wrapping around the Galactic centre. On
the right, the same stars in **GD1 stream coordinates** `(phi1, phi2)`: a
rotated sky frame in which the stream lies flat along `phi1`. That rotation
(`sm.stars_to_gd1`) is a fixed coordinate change we do in plain numpy — see the
note in the next section on why that matters for speed.

In [ ]:
sp.plot_stream_orbit_and_sky(stars, title="one GD1 stream at the fiducial parameters")
plt.show()

---
## 1 — From a variable star list to a fixed-size image

A network cannot read a list whose length changes every simulation. So, exactly
as `albatross` does, we turn each stream into a **fixed-shape image**:

1. rotate to GD1 observables — `(dist, phi1, phi2, vrad, pm_phi1_cosphi2, pm_phi2)`;
2. add per-observable Gaussian measurement errors and drop a few stars (`add_noise`);
3. add a uniform foreground of Milky-Way field stars (`sample_background`);
4. bin into **three 2-D histograms** — `(phi1, phi2)`, the proper motions, and
   `(dist, vrad)` — stacked into a single `(3, nbins, nbins)` array (`bin_stream`).

That fixed `(3, 48, 48)` shape is the whole trick: no matter how many stars a
stream has, the data the network sees is the same size.

> **Why this is fast enough to do live.** `albatross` rotates to GD1 with two
> jitted `jax` vmaps. Because `N_stars` changes every call, those vmaps
> *re-compile every single simulation* — several seconds each. `sm.stars_to_gd1`
> reimplements the identical rotation in numpy (it is affine — a rotation plus
> unit conversions), reproducing `sstrax` to float precision at ~0.5 ms. That
> ~6× speedup is what makes a full training run feasible in a hackathon.
> The timing test is in `hackathon_solutions/` (solutions branch).

In [ ]:
rng = np.random.default_rng(SEED)
truth_full = [sm.TRUE_VALUES[k] for k in sm.TRUE_VALUES]         # all 16 at truth
image = sm.simulate_image(truth_full, infer_params=list(sm.TRUE_VALUES), rng=rng)
print("image shape:", image.shape, " counts per channel:", image.sum((1,2)).astype(int))
sp.plot_channels(image, title="the three data channels a stream maps to")
plt.show()

### ✏️ Exercise 1 — watch the parameters move the data *(worked warm-up)*

Before inferring anything, get a feel for what is *learnable*: does changing a
parameter visibly change the data? We do this one together — it is the pattern
every idea below builds on. Simulate the stream image at **two disruption ages**
(1000 vs 4000 Myr), holding everything else at truth, and compare. The older
stream has had longer to disrupt, so it is **longer along `phi1`** and its
kinematic channels shift too.

`sm.simulate_image(z, infer_params=names, rng=...)` takes a value vector `z`
paired with parameter `names`; here we vary just `"age"`.

In [ ]:
for age in [1000., 4000.]:
    img = sm.simulate_image([age], infer_params=["age"], rng=np.random.default_rng(1))
    sp.plot_channels(img, title=f"age = {age:.0f} Myr")
    plt.show()

The image moves clearly with `age` — so `age` is something the data can
constrain. Try the same with another parameter (mass `logmsat`, or a velocity
component `vxc`) to build a mental map of which parameters the stream "sees"
before you spend simulations inferring them.

---
## 2 — The forward model as a `falcon` graph *(worked)*

Same shape as notebook 2. The graph has two nodes:

- **`z`** — the parameters we infer, with a `falcon.priors.Product` prior and a
  `falcon.estimators.Flow`. Its `embedding` is `sm.StreamCNN`, a small 2-D CNN
  that compresses the `(3, 48, 48)` image to a feature vector.
- **`x`** — the data node, produced by `sm.StreamImage` running the full forward
  model on `z` (`parents: [z]`), with `observed:` pointing at an image on disk.

We start with the smallest interesting inference: the progenitor's **age and
mass**, `sm.DEFAULT_INFER == ['age', 'logmsat']`. First, save the observation.

In [ ]:
z_truth = [sm.TRUE_VALUES[p] for p in sm.DEFAULT_INFER]     # [age, logmsat] at truth
x_obs = sm.simulate_image(z_truth, rng=np.random.default_rng(42))
np.save("obs_stream.npy", x_obs.astype(np.float32))
print("saved obs_stream.npy", x_obs.shape, "| inferring", sm.DEFAULT_INFER,
      "| truth", z_truth)

The filled config: two `Product` priors over `(age, logmsat)` on the
`sm.PRIOR_RANGES` ranges, a `Flow` with the `StreamCNN` embedding, and the data
node pointed at the file we just saved.

In [ ]:
%%writefile config_streams.yml
logging:
  wandb: {enabled: false}
  local: {enabled: true}

paths:
  imports: ["."]                    # import streams_model.py from the working dir

buffer:
  min_samples: 256                  # sstrax is ~0.5-1 s/sim, so keep the budget modest
  max_samples: 512
  validation_samples: 96
  simulate_count: 256
  simulate_when_full: false

graph:
  z:
    evidence: [x]
    simulator:
      _target_: falcon.priors.Product
      priors:
        - ['uniform', 500.0, 5000.0]   # age      [Myr]
        - ['uniform', 3.0, 4.5]        # logmsat
    estimator:
      _target_: falcon.estimators.Flow
      max_epochs: 40
      net_type: nsf
      lr: 0.003
      gamma: 0.5
      embedding:                    # 2-D CNN over the (3, nbins, nbins) image
        _target_: streams_model.StreamCNN
        out_features: 16
        _input_: [x]
      batch_size: 64
      early_stop_patience: 20
      theta_norm: true
      use_best_models: true
    ray:
      num_gpus: 0

  x:
    parents: [z]
    simulator:
      _target_: streams_model.StreamImage    # infer_params defaults to sm.DEFAULT_INFER
    observed: "./obs_stream.npy"

sample:
  posterior:
    n: 800

**The YAML is the graphical model.** Ask falcon to draw it:

In [ ]:
!falcon graph -c config_streams.yml

---
## 3 — Train, and read out a posterior *(worked)*

`launch` simulates ↔ trains the flow, then we draw posterior samples. The cost
is the **simulations** — a few hundred at ~0.5–1 s each, so expect **~10–15
minutes** on a Colab CPU.

In [ ]:
!falcon launch -c config_streams.yml -o output/run_streams --no-interactive
!falcon sample posterior -c config_streams.yml -o output/run_streams

In [ ]:
def load_falcon_posterior(run_dir):
    """Stack the per-sample NPZs falcon writes (one (D,) array under key 'z')."""
    import glob
    files = sorted(glob.glob(f"{run_dir}/samples/posterior/*.npz"))
    return np.concatenate([np.atleast_2d(np.load(f)["z"]) for f in files], axis=0)


post = load_falcon_posterior("output/run_streams")
truth = [sm.TRUE_VALUES[p] for p in sm.DEFAULT_INFER]
print("posterior:", post.shape)
for i, p in enumerate(sm.DEFAULT_INFER):
    print(f"  {p:8s}: {post[:, i].mean():9.3f} +/- {post[:, i].std():7.3f}   (truth {truth[i]})")
sp.plot_posterior(post, sm.DEFAULT_INFER, truth=truth, title="falcon posterior vs truth")
plt.show()

On the build machine this recovers `age = 2944 ± 171` (truth 3000) and
`logmsat = 4.076 ± 0.125` (truth 4.05) — the truth sits inside the posterior,
with the physical age–mass degeneracy visible (a longer stream can mean older
*or* heavier). This is the baseline the student notebook asks attendees to reach
on their own; everything in the open-ended list below extends it.